# Rice Disease Detection — AI AgriData Intelligence Race (TELEPATI 8.0)

Notebook ini memuat seluruh alur pengembangan model deteksi penyakit padi, mulai dari dataset mentah resmi sampai inference dan perhitungan skor.

| | |
|---|---|
| Tugas | Object detection 11 kelas (10 penyakit + Healthy), citra 640×640 |
| Data | Dataset resmi Agritech Data (format COCO, split train/valid/test) |
| Model | Ultralytics YOLO, dibangun dari file `.yaml` dan dilatih dari nol |
| Metrik | F1-Score makro pada split test resmi |
| Platform | Google Colab, GPU T4 |

**Mode eksekusi**

| `MODE` | Yang dijalankan | Kegunaan |
|---|---|---|
| `"full"` | Bagian 1–10 | Reproduksi lengkap: data mentah → training → evaluasi |
| `"inference_only"` | Bagian 1–3, lalu 11 | Reproduksi skor dari weights rilis, tanpa training |

**Kepatuhan aturan**

- Seluruh pemrosesan data dimulai dari zip dataset mentah dan dikerjakan di notebook ini.
- Training hanya memakai split **train**. Split **valid** dipakai untuk memilih checkpoint. Split **test** hanya dipakai di bagian 10 (sekali) dan bagian 11.
- Arsitektur dibangun dari file `.yaml` dengan `pretrained=False`. Tidak ada bobot eksternal yang dipakai untuk training.
- Seed tetap (`SEED = 42`) dan mode deterministik di setiap tahap.

## 0. Konfigurasi

Semua parameter yang perlu diubah ada di sel ini. Sel lain tidak perlu disunting.

- `RUN_ID` memilih konfigurasi training dari tabel `RUNS` di bagian 7.
- `WEIGHTS_URL`, `WEIGHTS_SHA256`, dan `E2_CONF` diisi setelah model final dirilis.
- `RUN_FINAL_TEST` dibiarkan `False` sampai model final dipilih dari hasil valid.

In [ ]:
MODE = "full"  # "full" | "inference_only"
RUN_ID = "A_yolo11s_s42"

DATA_ZIP = "/content/drive/MyDrive/telepati/data/telepati-agridata-raw.zip"
DATA_ZIP_SHA256 = "797e47749f91f4403e778dd0150716da1cd94cddbecee2ec56cf5545c50cec50"

RUNS_DIR = "/content/drive/MyDrive/telepati/runs"
WORK_DIR = "/content/work"

WEIGHTS_URL = ""
WEIGHTS_SHA256 = ""
E2_CONF = None

SEED = 42
IMGSZ = 640
EVAL_BATCH = 16
RUN_FINAL_TEST = False

## 1. Lingkungan

Memasang versi library yang dipin, menghubungkan Google Drive (khusus Colab), lalu mengunci seed.

Catatan: saat training dengan AMP, Ultralytics mengunduh model kecil hanya untuk *memeriksa* dukungan mixed precision pada GPU. Bobot tersebut tidak dipakai sebagai inisialisasi; model tetap dibangun dari `.yaml`.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.184"], check=True)
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import hashlib
import json
import math
import os
import random
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

WORK_DIR = Path(WORK_DIR)
RUNS_DIR = Path(RUNS_DIR)
RAW_DIR = WORK_DIR / "raw"
YOLO_DIR = WORK_DIR / "yolo"
SPLITS = ("train", "valid", "test")

print(f"python      {sys.version.split()[0]}")
print(f"torch       {torch.__version__}")
print(f"ultralytics {ultralytics.__version__}")
print(f"device      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## 2. Dataset mentah

Zip berisi folder `train/`, `valid/`, dan `test/` persis seperti dataset resmi (gambar `.jpg` + `_annotations.coco.json`). Checksum diverifikasi sebelum ekstraksi supaya data yang dipakai identik dengan data resmi.

In [ ]:
EXPECTED_COUNTS = {
    "train": {"images": 10133, "annotations": 20163},
    "valid": {"images": 2106, "annotations": 4888},
    "test": {"images": 1059, "annotations": 2670},
}


def sha256sum(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_dataset(zip_path: Path, dest: Path, expected_sha256: str) -> None:
    actual = sha256sum(zip_path)
    if actual != expected_sha256:
        raise ValueError(f"Checksum dataset tidak cocok: {actual}")
    if all((dest / split / "_annotations.coco.json").exists() for split in SPLITS):
        print(f"Dataset sudah terekstrak di {dest}")
        return
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    print(f"Dataset diekstrak ke {dest}")


def load_coco(split: str) -> dict:
    with open(RAW_DIR / split / "_annotations.coco.json", encoding="utf-8") as fh:
        return json.load(fh)


extract_dataset(Path(DATA_ZIP), RAW_DIR, DATA_ZIP_SHA256)
coco = {split: load_coco(split) for split in SPLITS}

for split, expected in EXPECTED_COUNTS.items():
    n_images, n_annotations = len(coco[split]["images"]), len(coco[split]["annotations"])
    assert n_images == expected["images"], f"{split}: {n_images} gambar"
    assert n_annotations == expected["annotations"], f"{split}: {n_annotations} anotasi"
    print(f"{split:<6} {n_images:>6} gambar  {n_annotations:>6} anotasi")

## 3. Pemetaan 21 kategori → 11 kelas canonical

File COCO bawaan memiliki 21 kategori: 18 nama kelas dengan variasi penulisan dan 3 supercategory (`Leaf-blight`, `Rice-Leaf-Diseasee`, `paddy`) yang tidak memiliki bounding box. Pemetaan dilakukan berdasarkan nama kategori, lalu divalidasi agar tidak ada anotasi yang tertinggal.

Urutan kelas mengikuti urutan alfabet pada halaman regulasi dan dipakai konsisten di training, evaluasi, dan inference.

In [ ]:
CLASS_NAMES = [
    "Bacterial leaf blight",
    "Bacterial panicle blight",
    "Blast",
    "Brown spot",
    "False smut",
    "Healthy",
    "Leaf roller",
    "Leaf scald",
    "Narrow brown",
    "Sheath blight",
    "Tungro",
]

RAW_TO_CANONICAL = {
    "Bacterial leaf blight": "Bacterial leaf blight",
    "Bacterial panicle Blight": "Bacterial panicle blight",
    "Blast": "Blast",
    "Leaf blast": "Blast",
    "Infected Blast": "Blast",
    "BrownSpot": "Brown spot",
    "Brown spot": "Brown spot",
    "False-Smut": "False smut",
    "Healthy Rice Leaf": "Healthy",
    "Healthy Rice beads": "Healthy",
    "Healthy": "Healthy",
    "healthy": "Healthy",
    "Leaf-roller": "Leaf roller",
    "Leaf Scald": "Leaf scald",
    "Leaf scald": "Leaf scald",
    "Narrow brown": "Narrow brown",
    "Sheath Blight": "Sheath blight",
    "Rice-Tungro": "Tungro",
}
SUPERCATEGORIES = {"Leaf-blight", "Rice-Leaf-Diseasee", "paddy"}
CLASS_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def to_frames(data: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Returns (images, boxes); boxes use canonical class ids and COCO xywh in pixels.
    category_names = {c["id"]: c["name"] for c in data["categories"]}
    unknown = set(category_names.values()) - set(RAW_TO_CANONICAL) - SUPERCATEGORIES
    if unknown:
        raise ValueError(f"Kategori tanpa pemetaan: {sorted(unknown)}")

    images = pd.DataFrame(data["images"])[["id", "file_name", "width", "height"]]
    rows = []
    for ann in data["annotations"]:
        raw_name = category_names[ann["category_id"]]
        if raw_name in SUPERCATEGORIES:
            continue
        x, y, w, h = ann["bbox"]
        rows.append((ann["image_id"], CLASS_INDEX[RAW_TO_CANONICAL[raw_name]], x, y, w, h))
    boxes = pd.DataFrame(rows, columns=["image_id", "class_id", "x", "y", "w", "h"])
    boxes = boxes.merge(images, left_on="image_id", right_on="id").drop(columns="id")
    return images, boxes


frames = {split: to_frames(coco[split]) for split in SPLITS}

for split in SPLITS:
    _, boxes = frames[split]
    assert len(boxes) == EXPECTED_COUNTS[split]["annotations"], split

class_counts = pd.DataFrame(
    {split: frames[split][1]["class_id"].value_counts().sort_index() for split in SPLITS}
).set_axis(CLASS_NAMES)
class_counts.loc["Total"] = class_counts.sum()
class_counts

## 4. Pembersihan anotasi

Pembersihan hanya diterapkan pada data **train**. Label valid dan test dipakai apa adanya (hanya dipetakan ke 11 kelas) supaya skor yang kita hitung sama dengan perhitungan juri.

| Aturan | Tindakan |
|---|---|
| Koordinat di luar batas gambar | Dipotong ke batas gambar |
| Sisi box < 2 px | Dibuang (luas 1–3 px², hampir pasti label rusak) |
| Box identik dalam satu gambar | Disisakan satu |
| Box seluas gambar penuh | Dipertahankan (gaya label sebagian sumber data) |
| Gambar tanpa anotasi | Dipertahankan sebagai gambar latar |

In [ ]:
MIN_BOX_SIDE = 2.0


def clip_boxes(boxes: pd.DataFrame) -> pd.DataFrame:
    out = boxes.copy()
    x1 = out["x"].clip(lower=0)
    y1 = out["y"].clip(lower=0)
    x2 = (out["x"] + out["w"]).clip(upper=out["width"])
    y2 = (out["y"] + out["h"]).clip(upper=out["height"])
    out["x"], out["y"], out["w"], out["h"] = x1, y1, x2 - x1, y2 - y1
    return out


def clean_train_boxes(boxes: pd.DataFrame) -> pd.DataFrame:
    clipped = clip_boxes(boxes)
    too_small = (clipped["w"] < MIN_BOX_SIDE) | (clipped["h"] < MIN_BOX_SIDE)
    kept = clipped[~too_small]
    deduped = kept.drop_duplicates(subset=["image_id", "class_id", "x", "y", "w", "h"])
    print(f"train: {len(boxes)} box → {len(deduped)} box "
          f"(sisi < {MIN_BOX_SIDE:g}px: {int(too_small.sum())}, duplikat: {len(kept) - len(deduped)})")
    return deduped


if MODE == "full":
    train_images, train_boxes = frames["train"]
    train_boxes = clean_train_boxes(train_boxes)
    valid_images, valid_boxes = frames["valid"][0], clip_boxes(frames["valid"][1])

## 5. Konversi ke format YOLO

Setiap gambar ditautkan (hard link, atau disalin bila tidak didukung) ke struktur `images/<split>` dan labelnya ditulis ke `labels/<split>` dalam format `class cx cy w h` ternormalisasi.

Untuk run dengan penyeimbangan kelas, gambar train diulang menurut *repeat factor sampling*: kelas yang muncul di sedikit gambar mendapat faktor `r_c = max(1, sqrt(t / f_c))`, dan setiap gambar diulang sebanyak faktor terbesar dari kelas di dalamnya. Pembulatan memakai RNG dengan seed tetap. Hanya data train yang dipakai untuk menghitung faktor ini.

In [ ]:
def link_or_copy(src: Path, dst: Path) -> None:
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def yolo_lines(image_boxes: pd.DataFrame) -> list[str]:
    lines = []
    for row in image_boxes.itertuples(index=False):
        cx = (row.x + row.w / 2) / row.width
        cy = (row.y + row.h / 2) / row.height
        lines.append(f"{row.class_id} {cx:.6f} {cy:.6f} {row.w / row.width:.6f} {row.h / row.height:.6f}")
    return lines


def export_split(split: str, images: pd.DataFrame, boxes: pd.DataFrame, out_root: Path,
                 repeats: dict[str, int] | None = None) -> int:
    image_dir, label_dir = out_root / "images" / split, out_root / "labels" / split
    for folder in (image_dir, label_dir):
        shutil.rmtree(folder, ignore_errors=True)
        folder.mkdir(parents=True)

    boxes_by_file = dict(tuple(boxes.groupby("file_name")))
    empty = boxes.iloc[0:0]
    written = 0
    for file_name in sorted(images["file_name"]):
        lines = "\n".join(yolo_lines(boxes_by_file.get(file_name, empty)))
        stem, suffix = Path(file_name).stem, Path(file_name).suffix
        for k in range((repeats or {}).get(file_name, 1)):
            name = stem if k == 0 else f"{stem}__rep{k}"
            link_or_copy(RAW_DIR / split / file_name, image_dir / f"{name}{suffix}")
            (label_dir / f"{name}.txt").write_text(lines)
            written += 1
    return written


def repeat_factors(images: pd.DataFrame, boxes: pd.DataFrame, threshold: float, seed: int) -> dict[str, int]:
    classes_per_image = boxes.groupby("file_name")["class_id"].agg(set)
    image_freq = {c: sum(c in s for s in classes_per_image) / len(images) for c in range(len(CLASS_NAMES))}
    class_factor = {c: max(1.0, math.sqrt(threshold / f)) if f > 0 else 1.0 for c, f in image_freq.items()}

    rng = np.random.default_rng(seed)
    repeats = {}
    for file_name in sorted(images["file_name"]):
        classes = classes_per_image.get(file_name, set())
        factor = max((class_factor[c] for c in classes), default=1.0)
        repeats[file_name] = int(factor) + int(rng.random() < factor - int(factor))
    return repeats

## 6. Konfigurasi dataset

`data.yaml` hanya berisi train dan valid. Test ditulis ke file terpisah (`data_test.yaml`) di bagian 10–11, sehingga test tidak pernah terdaftar di konfigurasi training.

In [ ]:
def write_data_yaml(path: Path, root: Path, val_split: str = "valid") -> Path:
    config = {
        "path": str(root),
        "train": "images/train",
        "val": f"images/{val_split}",
        "names": dict(enumerate(CLASS_NAMES)),
    }
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path